# Project 4 — Text-to-SQL with guardrails
**Track A (local Ollama).** NL questions → runnable SQL over a toy DB; refuse destructive requests.
**Data:** `data/shop.sql` (customers + orders) · `data/sql_questions.jsonl` · `data/sql_safety.jsonl`.
**Evaluated on:** execution accuracy + a safety set that must be blocked.

In [ ]:
import sys, json; sys.path.append("../..")   # import utils/ and eval/ from repo root
import sqlite3
from utils import ask

con = sqlite3.connect(":memory:")
con.executescript(open("data/shop.sql", encoding="utf-8").read())
SCHEMA = "customers(id, name, city); orders(id, customer_id, product, amount)"
QS = [json.loads(l) for l in open("data/sql_questions.jsonl", encoding="utf-8")]
SAFETY = [json.loads(l) for l in open("data/sql_safety.jsonl", encoding="utf-8")]
print(len(QS), "questions;", len(SAFETY), "safety cases")

## Starter: schema-aware, read-only prompt

In [ ]:
def nl_to_sql(question):
    prompt = (f"Schema:\n{SCHEMA}\n\n"
              "Write ONE read-only SQLite SELECT for the question. Return ONLY the SQL. "
              "Never write INSERT/UPDATE/DELETE/DROP.\n\n"
              f"Question: {question}")
    return ask(prompt).strip().strip("`").rstrip(";")

def run(sql):
    return con.execute(sql).fetchall()

q = QS[0]
print("Q:", q["question"])
print("model SQL:", nl_to_sql(q["question"]))
print("reference rows:", run(q["reference_sql"]))

## Execution-accuracy check (compare to the reference query's rows)

In [ ]:
def exec_accuracy(items):
    ok = 0
    for it in items:
        try:
            if run(nl_to_sql(it["question"])) == run(it["reference_sql"]):
                ok += 1
        except Exception:
            pass
    print(f"execution accuracy: {ok}/{len(items)} = {ok/len(items):.0%}")

exec_accuracy(QS)

## Your tasks
1. Add a guard that rejects any non-SELECT before it runs.
2. Confirm every safety request is refused, not executed.
3. Grow the schema and question set.

In [ ]:
# SAFETY — these must be blocked
for s in SAFETY:
    print(s['request'], '->', nl_to_sql(s['request']))
# TODO: your code here
